# Pretrain the candle language model on Kaggle (GPU)

Builds `wav-transformer` with `--features candle-cuda` and pretrains the complex-valued candle transformer
(width 256, 12 layers, attention window 256, ~63 M parameters, ~25 M in the layers) on German Wikipedia.

**Once:** Kaggle account with phone verification (internet), *File → Import Notebook* with this file,
*Session options*: **Accelerator GPU T4 x2** (or P100), **Internet on**.

**First run:** `RESUME = False`. The notebook downloads `WIKI_PARTS` of German Wikipedia from Hugging Face, tokenizes
them into token shards (~150 M tokens per part, ~2 minutes each) and keeps the shards, the binary and the model in the
**Output**.

**Every further run:** *Add Input → Your Work → Notebooks →* this notebook (its last output), `RESUME = True`,
*Save Version → Save & Run All*. The shards, the binary (unless the Rust code changed) and the model are taken from the
input; only a model with a better validation loss replaces the saved one.

`MAX_HOURS` stops the training in time (Kaggle ends a session after 12 h). The GPU quota is ~30 h per week.
If the GPU runs out of memory, lower `MICRO_BATCH` (sequences per pass; the batch stays `BATCH_SIZE`).


In [ ]:
# ── Settings ─────────────────────────────────────────────────────────────
REPO = "https://github.com/RodionKovalenko/rustCrates"
BRANCH = "feature/neural-network-rust"
WIKI_PARTS = "0-7"        # Wikipedia parquet parts (0..19), ~150 M tokens each; only used when no shards are in the input
STEPS = 7000             # optimizer steps of this run (~5.5 s each on a T4: about 11 h; the learning rate decays over them)
BATCH_SIZE = 16          # sequences of 512 tokens per step
MICRO_BATCH = 16         # sequences per forward/backward pass (halved automatically if the GPU runs out of memory)
LEARNING_RATE = 0.0003   # 6e-4 made the loss rise again on the first T4 run
RESUME = False           # continue the model found in the notebook input
WINDOW = 256             # attention window of a new model (tokens each token sees back)
MAX_HOURS = 11.0

INPUT_ROOT = "/kaggle/input"
OUTPUT_ROOT = "/kaggle/working"
SOURCE_DIR = "/tmp/rustCrates"


In [ ]:
import glob, json, os, shutil, subprocess, sys, time, zipfile

def run(cmd, cwd=None, env=None):
    print("$", cmd, flush=True)
    subprocess.run(cmd, shell=True, check=True, cwd=cwd, env=env)

if subprocess.run("nvidia-smi", shell=True).returncode != 0:
    raise SystemExit("No GPU: set Session options -> Accelerator -> GPU T4 x2 (Save Version uses the notebook's accelerator), then run again.")

# Rust (the repository's rust-toolchain file selects the version on the first cargo call)
if not os.path.exists(os.path.expanduser("~/.cargo/bin/cargo")):
    run("curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal")
os.environ["PATH"] = os.path.expanduser("~/.cargo/bin") + os.pathsep + os.environ["PATH"]

# Source
if os.path.exists(SOURCE_DIR):
    run(f"git fetch --depth 200 origin {BRANCH} && git checkout -B {BRANCH} FETCH_HEAD", cwd=SOURCE_DIR)
else:
    run(f"git clone --depth 200 --branch {BRANCH} {REPO} {SOURCE_DIR}")
run("git log -1 --oneline", cwd=SOURCE_DIR)


In [ ]:
# CUDA toolkit: nvcc compiles candle's kernels, cuBLAS / cuRAND / NVRTC are linked. Kaggle images do not always
# ship the full toolkit; then its packages are taken from NVIDIA's apt repository (unpacked, ~3 GB, a few minutes).
import re, urllib.request

CUDA_VERSION = "12-8"
CUDA_PACKAGES = ["cuda-nvcc", "cuda-nvvm", "cuda-crt", "cuda-cudart", "cuda-cudart-dev", "cuda-cccl", "cuda-driver-dev",
                 "libcublas", "libcublas-dev", "libcurand", "libcurand-dev", "cuda-nvrtc", "cuda-nvrtc-dev"]

def find_cuda_root():
    for root in [os.environ.get("CUDA_HOME", ""), "/usr/local/cuda"] + sorted(glob.glob("/usr/local/cuda-*"), reverse=True):
        if root and os.path.exists(os.path.join(root, "bin", "nvcc")) and glob.glob(os.path.join(root, "**", "libcublas.so*"), recursive=True):
            return root
    return None

def version_key(name):
    return [int(x) if x.isdigit() else x for x in re.split(r"[._~-]", name)]

def install_cuda_toolkit(target="/tmp/cuda_toolkit"):
    release = [l.split("=")[1].strip().strip('"') for l in open("/etc/os-release") if l.startswith("VERSION_ID=")][0]
    repo = f"https://developer.download.nvidia.com/compute/cuda/repos/ubuntu{release.replace('.', '')}/x86_64"
    listing = urllib.request.urlopen(repo + "/").read().decode()
    os.makedirs(target, exist_ok=True)
    for package in CUDA_PACKAGES:
        name = f"{package}-{CUDA_VERSION}"
        debs = sorted(set(re.findall(re.escape(name) + r"_[0-9][^\"'<>/]*_amd64\.deb", listing)), key=version_key)
        if not debs:
            raise RuntimeError(f"{name} not found in {repo}")
        run(f"curl -sSfL -o /tmp/pkg.deb {repo}/{debs[-1]} && dpkg -x /tmp/pkg.deb {target} && rm /tmp/pkg.deb")
    return glob.glob(os.path.join(target, "usr", "local", "cuda-*"))[0]

LINK_LIBS = ["cuda", "nvrtc", "curand", "cublas", "cublasLt", "cudart"]
LINK_DIR = "/tmp/cuda_link"

def link_dirs():
    lib = os.path.join(CUDA_ROOT, "targets", "x86_64-linux", "lib")
    return [lib, os.path.join(lib, "stubs"), os.path.join(CUDA_ROOT, "lib64"), os.path.join(CUDA_ROOT, "lib64", "stubs"),
            "/usr/lib/x86_64-linux-gnu", "/usr/lib64", "/usr/local/nvidia/lib64", "/usr/local/nvidia/lib"]

def prepare_link_dir():
    """The linker needs libX.so; driver and toolkit often only ship libX.so.N. Missing ones are linked here."""
    os.makedirs(LINK_DIR, exist_ok=True)
    known = subprocess.run("ldconfig -p", shell=True, capture_output=True, text=True).stdout
    for name in LINK_LIBS:
        if any(os.path.exists(os.path.join(d, f"lib{name}.so")) for d in link_dirs()):
            continue
        candidates = [p for d in link_dirs() for p in sorted(glob.glob(os.path.join(d, f"lib{name}.so.*")))]
        candidates += re.findall(rf"=> (\S*/lib{re.escape(name)}\.so\.[0-9.]*)", known)
        if not candidates:
            print(f"warning: lib{name}.so not found")
            continue
        target = os.path.join(LINK_DIR, f"lib{name}.so")
        if os.path.lexists(target):
            os.remove(target)
        os.symlink(candidates[0], target)
        print(f"lib{name}.so -> {candidates[0]}")

def cuda_lib_path():
    return os.pathsep.join([LINK_DIR] + link_dirs() + [os.environ.get("LIBRARY_PATH", "")])

CUDA_ROOT = find_cuda_root()
if CUDA_ROOT is None:
    print("No complete CUDA toolkit found; installing it")
    CUDA_ROOT = install_cuda_toolkit()
os.environ["PATH"] = os.path.join(CUDA_ROOT, "bin") + os.pathsep + os.environ["PATH"]
# The program loads cuBLAS etc. at run time; the toolkit's libraries come after the system ones (the driver's libcuda).
os.environ["LD_LIBRARY_PATH"] = os.environ.get("LD_LIBRARY_PATH", "") + os.pathsep + os.path.join(CUDA_ROOT, "targets", "x86_64-linux", "lib")
run("nvcc --version | tail -1")
prepare_link_dir()

# Build (~30 minutes with CUDA), or reuse the binary of an earlier output when the code is unchanged
features = "--features candle-cuda"
binary = os.path.join(SOURCE_DIR, "target", "release", "wav-transformer")
commit = subprocess.run("git rev-parse HEAD", shell=True, cwd=SOURCE_DIR, capture_output=True, text=True).stdout.strip()
BUILD_INPUTS = "src Cargo.toml Cargo.lock build.rs rust-toolchain"

def reusable_binary():
    for info_path in sorted(glob.glob(os.path.join(INPUT_ROOT, "**", "bin", "build.json"), recursive=True)):
        with open(info_path) as f:
            info = json.load(f)
        cached = os.path.join(os.path.dirname(info_path), "wav-transformer")
        if info.get("features") != features or not os.path.exists(cached):
            continue
        if info.get("commit") == commit or subprocess.run(f"git diff --quiet {info.get('commit')} HEAD -- {BUILD_INPUTS}", shell=True, cwd=SOURCE_DIR).returncode == 0:
            return cached, info
        print(f"Cached binary of {info.get('commit', '?')[:7]} is outdated")
    return None, None

cached, info = reusable_binary()
if cached:
    os.makedirs(os.path.dirname(binary), exist_ok=True)
    shutil.copy(cached, binary)
    os.chmod(binary, 0o755)
    print(f"Using the cached binary built from {info['commit'][:7]}; no build needed")
else:
    # candle compiles its CUDA kernels for the GPU's compute capability (T4: 75, P100: 60).
    cap = subprocess.run("nvidia-smi --query-gpu=compute_cap --format=csv,noheader", shell=True, capture_output=True, text=True).stdout.split()
    env = dict(os.environ, CUDA_ROOT=CUDA_ROOT, CUDA_PATH=CUDA_ROOT, CUDA_HOME=CUDA_ROOT, LIBRARY_PATH=cuda_lib_path(),
               RUSTFLAGS=f"-L native={LINK_DIR}")  # the linker finds libcuda.so etc. in LINK_DIR
    if cap:
        env["CUDA_COMPUTE_CAP"] = cap[0].replace(".", "")
    run(f"cargo build --release {features}", cwd=SOURCE_DIR, env=env)
    info = {"commit": commit, "features": features}

bin_dir = os.path.join(OUTPUT_ROOT, "bin")
os.makedirs(bin_dir, exist_ok=True)
shutil.copy(binary, os.path.join(bin_dir, "wav-transformer"))
with open(os.path.join(bin_dir, "build.json"), "w") as f:
    json.dump(info, f)


In [ ]:
# Pretraining data: token shards from an earlier output, or new ones from Wikipedia
def find_shards():
    for meta in sorted(glob.glob(os.path.join(INPUT_ROOT, "**", "pretrain", "meta.json"), recursive=True)):
        return os.path.dirname(meta)
    return None

DATA_DIR = find_shards()
if DATA_DIR:
    print("Using the token shards in", DATA_DIR)
else:
    DATA_DIR = os.path.join(OUTPUT_ROOT, "pretrain")
    run(f"pip install -q tokenizers pyarrow")
    run(f"python kaggle/prepare_pretrain.py --out {DATA_DIR} --parts {WIKI_PARTS} --download-dir /tmp/wiki_download", cwd=SOURCE_DIR)
    shutil.rmtree("/tmp/wiki_download", ignore_errors=True)
with open(os.path.join(DATA_DIR, "meta.json")) as f:
    meta = json.load(f)
print(f"{meta['train_tokens'] / 1e6:.0f} M training tokens, {meta['val_tokens'] / 1e6:.1f} M validation tokens")


In [ ]:
# Previous model (Add Input of an earlier output)
CHECKPOINT = "candle_lm.safetensors"
storage = os.path.join(SOURCE_DIR, "STORAGE")
os.makedirs(storage, exist_ok=True)
previous = None
if RESUME:
    for path in sorted(glob.glob(os.path.join(INPUT_ROOT, "**", "model", CHECKPOINT), recursive=True)):
        previous = os.path.dirname(path)
        break
if previous:
    for name in os.listdir(previous):
        if name.startswith(CHECKPOINT):
            shutil.copy(os.path.join(previous, name), storage)
    print("Continuing the model from", previous)
else:
    print("Training a new model" + (" (no previous model in the input)" if RESUME else ""))
MODE = "resume" if previous else "new"


In [ ]:
# Pretrain; the full log is kept in the output as pretrain.log
# The GPU driver compiles candle's kernels on first use (~10 minutes); its cache goes to the output and is reused.
CUDA_CACHE = os.path.join(OUTPUT_ROOT, "cuda_cache")
for cached_jit in sorted(glob.glob(os.path.join(INPUT_ROOT, "**", "cuda_cache"), recursive=True)):
    if not os.path.exists(CUDA_CACHE):
        shutil.copytree(cached_jit, CUDA_CACHE)
        print("Using the kernel cache from", cached_jit)
    break
env = dict(os.environ, TRAIN_MAX_HOURS=str(MAX_HOURS), MICRO_BATCH=str(MICRO_BATCH),
           CUDA_CACHE_PATH=CUDA_CACHE, CUDA_CACHE_MAXSIZE=str(4 << 30))
cmd = [binary, "pretrain", DATA_DIR, str(STEPS), str(BATCH_SIZE), str(LEARNING_RATE), MODE, str(WINDOW)]
print("$", " ".join(cmd), flush=True)
started = time.time()
with open(os.path.join(OUTPUT_ROOT, "pretrain.log"), "a") as log:
    proc = subprocess.Popen(cmd, cwd=SOURCE_DIR, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        log.write(line)
        log.flush()
        print(f"[{(time.time() - started) / 3600:5.2f} h] {line.rstrip()}", flush=True)
    proc.wait()
print("exit code", proc.returncode)


In [ ]:
# Output: the model (+ config, training state, validation history), a zip of it and the loss curve
output = os.path.join(OUTPUT_ROOT, "model")
os.makedirs(output, exist_ok=True)
for name in os.listdir(storage):
    if name.startswith(CHECKPOINT):
        shutil.copy(os.path.join(storage, name), output)
archive = shutil.make_archive(os.path.join(OUTPUT_ROOT, "candle_lm"), "zip", output)
print("Saved", archive, f"{os.path.getsize(archive) / 1e6:.1f} MB")

import csv
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
history = os.path.join(output, CHECKPOINT + ".history.csv")
rows = [r for r in csv.reader(open(history)) if len(r) >= 2] if os.path.exists(history) else []
if not rows:
    print("No validation results yet (see the training log above)")
else:
    steps, losses = [int(r[0]) for r in rows], [float(r[1]) for r in rows]
    fig, ax = plt.subplots(figsize=(10, 5.5), dpi=120)
    ax.plot(steps, losses, color="#e07b00", linewidth=2, label="Validation")
    best = min(range(len(losses)), key=losses.__getitem__)
    ax.scatter([steps[best]], [losses[best]], color="#e07b00", zorder=3)
    ax.annotate(f"best {losses[best]:.3f} (step {steps[best]})", (steps[best], losses[best]), textcoords="offset points", xytext=(0, 10), ha="center")
    ax.set_xlabel("Optimizer step"); ax.set_ylabel("Validation loss per token"); ax.set_title("candle language model")
    ax.grid(alpha=0.3); ax.legend(); fig.tight_layout()
    fig.savefig(os.path.join(OUTPUT_ROOT, "loss_curve.png")); plt.show()


In [ ]:
# Try the model: greedy continuation of a German text
if os.path.exists(os.path.join(storage, CHECKPOINT)):
    run(f'{binary} lm-generate 60 "Die Hauptstadt von Deutschland ist"', cwd=SOURCE_DIR, env=env)
else:
    print("No model saved yet")
